<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## 02-2 · Part 6: Gradients and Descent

**The gradient determines first-order score changes; a finite descent step still requires feasibility and score checks.**

Part 5 calculated gradients through the classroom simulation using Jacobians and the chain rule. This lecture retains that formulation and connects the gradient to directional derivatives, level-set geometry, and descent steps. Finite differences provide an independent numerical evaluation of the same score gradient; inner products and the Euclidean norm determine the local descent direction.

### 1 · Classroom formulation

The real decision is how much cooling to use early and late. More cooling can reduce heat discomfort, but it uses energy and may make the room too cold.

Let \\(u=(u_{\mathrm{early}},u_{\mathrm{late}})\\). The horizon has \\(n=12\\) decision steps, with actions \\(u_0,\ldots,u_{11}\\) and states \\(T_0,\ldots,T_{12}\\).

> $\displaystyle u_t=\begin{cases}u_{\mathrm{early}},&t=0,\ldots,5,\\u_{\mathrm{late}},&t=6,\ldots,11.\end{cases}$

Indoor temperature \\(T_t\\) is produced by the system; it is not chosen directly. With initial temperature \\(T_0=27\,^{\circ}\mathrm C\\), the physical transition is

> $\displaystyle T_{t+1}=F(T_t,T_t^{\mathrm{out}},N_t,u_t;a,b,c)$
>
> $\displaystyle \phantom{T_{t+1}}=T_t+a(T_t^{\mathrm{out}}-T_t)+bN_t-cu_t,\quad t=0,\ldots,11.$

| Role | Values held fixed in the demonstrations |
|:---|:---|
| Fixed parameters | $(a,b,c)=(0.12,0.012,0.45)$ |
| External inputs | $T_t^{\mathrm{out}}=31\,^{\circ}\mathrm C$ and $N_t=20$ people at every step |
| Cooling limits | $u_{\min}=0$, $u_{\max}=5$ cooling units |
| State limits | $T_{\min}=20\,^{\circ}\mathrm C$, $T_{\max}=30\,^{\circ}\mathrm C$ |
| Energy limit | $E_{\max}=60$ model energy units |
| Energy-weight hyperparameter | $\lambda_E=1$ unless stated otherwise |

The performance mapping \\(G\\) measures discomfort outside the 22–24 °C comfort range and energy use:

> $\displaystyle D(u)=\sum_{t=1}^{12}\left[\max(T_t-24,0)^2+\max(22-T_t,0)^2\right].$
>
> $\displaystyle E(u)=\frac12\sum_{t=0}^{11}u_t^2=3\left(u_{\mathrm{early}}^2+u_{\mathrm{late}}^2\right).$

\\(D\\) uses squared-temperature step units. \\(E\\) uses model energy units, not calibrated kWh. The weight converts energy into the chosen score scale:

> $\displaystyle J(u;\lambda_E)=H(D(u),E(u);\lambda_E)=D(u)+\lambda_EE(u).$

Feasibility requires cooling bounds, all state limits for \\(t=1,\ldots,12\\), and \\(E(u)\le E_{\max}\\). The comfort range is a performance target; the wider 20–30 °C range is a hard requirement.

In standard notation, \\(x=[u_{\mathrm{early}},u_{\mathrm{late}}]^{\mathsf T}\\) and \\(y=\operatorname{Sim}(x)\\). Here \\(\operatorname{Sim}\\) composes repeated \\(F\\) transitions with \\(G\\). The objective \\(f(y;\lambda_E)\\) is the standard-form name for the score supplied by \\(H\\). The score depends on the cooling decision through both the simulated states and energy use.

In [ ]:
import sys
import warnings

import matplotlib
import numpy as np
from matplotlib.patches import Patch, Rectangle
from matplotlib.lines import Line2D


def _pyplot(*, interactive=False):
    """Use ipympl outside the Playground, with a static fallback."""
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (ImportError, RuntimeError, ValueError):
            try:
                matplotlib.use("module://ipympl.backend_nbagg", force=True)
            except (ImportError, RuntimeError, ValueError):
                warnings.warn("Interactive backend unavailable; showing a static preview.")
    import matplotlib.pyplot as plt
    return plt


# Horizon and initial state
TIME_STEPS = 12
INITIAL_TEMPERATURE = 27.0

# Fixed parameters
WEATHER_EXCHANGE = 0.12
OCCUPANT_HEAT = 0.012
COOLING_EFFECT = 0.45

# External inputs
OUTSIDE_TEMPERATURE = np.full(TIME_STEPS, 31.0)
OCCUPANTS = np.full(TIME_STEPS, 20.0)

# Requirement limits
MIN_COOLING, MAX_COOLING = 0.0, 5.0
MIN_TEMPERATURE, MAX_TEMPERATURE = 20.0, 30.0
MAX_ENERGY = 60.0

# Evaluation hyperparameter
ENERGY_WEIGHT = 1.0

# Stable visual roles
BLUE, TEAL, ORANGE = "#2563EB", "#0F8B7C", "#E88726"
PURPLE, GRAY = "#7C3AED", "#9CA3AF"


def style_axis(axis):
    axis.grid(alpha=0.25)
    axis.spines[["top", "right"]].set_visible(False)


def decision_axes(axis):
    axis.set(xlabel="Early cooling (cooling units)",
             ylabel="Late cooling (cooling units)", xlim=(0, 5), ylim=(0, 5))
    axis.set_aspect("equal")
    style_axis(axis)

Simulation produces the state path. The performance mapping calculates discomfort and energy; the requirement checks determine feasibility. Evaluation collects these quantities with the score in one result.


In [ ]:
def expand_decision(decision):
    """Expand the chosen levels into u_0, ..., u_11."""
    decision = np.asarray(decision, dtype=float)
    if decision.shape != (2,) or not np.isfinite(decision).all():
        raise ValueError("A decision must contain two finite cooling levels.")
    return np.repeat(decision, TIME_STEPS // 2)


def simulate_classroom(decision):
    cooling_schedule = expand_decision(decision)
    temperatures = np.empty(TIME_STEPS + 1)
    temperatures[0] = INITIAL_TEMPERATURE
    for t in range(TIME_STEPS):
        temperatures[t + 1] = (
            temperatures[t]
            + WEATHER_EXCHANGE * (OUTSIDE_TEMPERATURE[t] - temperatures[t])
            + OCCUPANT_HEAT * OCCUPANTS[t]
            - COOLING_EFFECT * cooling_schedule[t]
        )
    return cooling_schedule, temperatures


def performance_outputs(cooling_schedule, temperatures):
    discomfort = np.sum(
        np.maximum(temperatures[1:] - 24.0, 0.0) ** 2
        + np.maximum(22.0 - temperatures[1:], 0.0) ** 2
    )
    energy = 0.5 * np.sum(cooling_schedule ** 2)
    return float(discomfort), float(energy)


def check_feasibility(decision, temperatures, energy):
    violations = []
    if not np.all((MIN_COOLING <= decision) & (decision <= MAX_COOLING)):
        violations.append("cooling bound")
    if np.min(temperatures[1:]) < MIN_TEMPERATURE:
        violations.append("minimum temperature")
    if np.max(temperatures[1:]) > MAX_TEMPERATURE:
        violations.append("maximum temperature")
    if energy > MAX_ENERGY:
        violations.append("energy limit")
    return tuple(violations)


def evaluate_candidate(decision, energy_weight=ENERGY_WEIGHT):
    decision = np.asarray(decision, dtype=float)
    cooling_schedule, temperatures = simulate_classroom(decision)
    discomfort, energy = performance_outputs(cooling_schedule, temperatures)
    violations = check_feasibility(decision, temperatures, energy)
    return {
        "decision": decision.copy(), "cooling_schedule": cooling_schedule,
        "temperatures": temperatures, "discomfort": discomfort, "energy": energy,
        "feasible": not violations, "violations": violations,
        "energy_weight": float(energy_weight),
        "objective": discomfort + energy_weight * energy,
    }


def score(decision, energy_weight=ENERGY_WEIGHT):
    """Also defined outside the feasible set for geometry and derivatives."""
    return evaluate_candidate(decision, energy_weight)["objective"]

### 2 · Partial derivatives and gradients

A derivative describes how an output changes per small input change. A **partial derivative** changes one coordinate and holds the others fixed. For early cooling,

> $\displaystyle \frac{\partial J}{\partial u_{\mathrm{early}}}(u;\lambda_E)=\lim_{\varepsilon\to0}\frac{J(u+\varepsilon e_1;\lambda_E)-J(u;\lambda_E)}{\varepsilon}.$

Here \\(e_1=(1,0)\\); using \\(e_2=(0,1)\\) gives the late-cooling partial derivative. The gradient puts the slopes in the same coordinate order as the decision:

> $\displaystyle \nabla_u J(u;\lambda_E)=\begin{bmatrix}\partial J/\partial u_{\mathrm{early}}\\\partial J/\partial u_{\mathrm{late}}\end{bmatrix}.$

For the classroom energy function, \\(E(u)=3(u_{\mathrm{early}}^2+u_{\mathrm{late}}^2)\\), so \\(\nabla_u E(u)=6u\\). At \\(u=(3,2)\\), its components are \\((18,12)\\). These are energy slopes, not full-score slopes. The discomfort contribution depends on the entire state path.

Part 5 obtained the full-score gradient analytically by propagating state sensitivities. Here the code evaluates the same derivative with a central finite difference:

> $\displaystyle [\nabla_u J(u;\lambda_E)]_i\approx\frac{J(u+\varepsilon e_i;\lambda_E)-J(u-\varepsilon e_i;\lambda_E)}{2\varepsilon},\quad i=1,2.$

The analyst chooses a small positive perturbation \\(\varepsilon\\). A large perturbation measures nonlocal behavior; an extremely small one can amplify floating-point error. Each perturbed score includes a fresh simulation.

The squared comfort penalties make \\(J\\) continuously differentiable here, including when temperature reaches 22 or 24 °C. Curvature can change at those thresholds. At a constraint boundary, the formula may evaluate mathematically defined but infeasible neighbors. These neighbors define the slope but remain infeasible candidates.

The two score slices each vary one cooling coordinate with the other fixed. Their tangent slopes are the components of the gradient at the same decision.

In [ ]:
def finite_difference_gradient(decision, energy_weight=ENERGY_WEIGHT, step=1e-4):
    decision = np.asarray(decision, dtype=float)
    if step <= 0:
        raise ValueError("The finite-difference step must be positive.")
    return np.array([
        (score(decision + step * direction, energy_weight)
         - score(decision - step * direction, energy_weight)) / (2 * step)
        for direction in np.eye(2)
    ])


def show_partial_derivatives(decision=(3.0, 2.0)):
    plt = _pyplot()
    decision = np.asarray(decision, dtype=float)
    gradient = finite_difference_gradient(decision)
    initial_score = score(decision)
    offsets = np.linspace(-0.45, 0.45, 121)
    figure, axes = plt.subplots(1, 2, figsize=(10.8, 4.4))
    for index, axis in enumerate(axes):
        direction = np.eye(2)[index]
        coordinates = decision[index] + offsets
        results = [evaluate_candidate(decision + offset * direction) for offset in offsets]
        actual = np.array([r["objective"] for r in results])
        feasible = np.array([r["feasible"] for r in results])
        axis.plot(coordinates, actual, color=GRAY, lw=2, label="Score extension")
        axis.plot(coordinates, np.where(feasible, actual, np.nan), color=BLUE,
                   lw=2.5, label="Feasible scores")
        axis.plot(coordinates, initial_score + offsets * gradient[index], "--",
                   color=ORANGE, label="Local tangent prediction")
        axis.scatter(decision[index], initial_score, color=ORANGE, zorder=4)
        name = ["Early", "Late"][index]
        axis.set(xlabel=f"{name} cooling (cooling units)", ylabel="Score J (score units)",
                 title=f"{name} slope = {gradient[index]:+.3f} score units / cooling unit")
        axis.legend(fontsize=8)
        style_axis(axis)
    figure.tight_layout()
    return figure

The early-cooling tangent slope is negative; the late-cooling slope is positive. Each tangent approximates its score slice near the orange point, with increasing error farther from that point.


<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-2_mathematics_for_optimization/assets/06_partial_derivatives.svg" alt="Score slices changing early and late cooling separately with their local tangent lines" width="900" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

In [ ]:
decision = np.array([3.0, 2.0])
print("Exact energy gradient:", 6 * decision)
for perturbation in [1e-2, 1e-3, 1e-4]:
    print(f"Full-score gradient, epsilon={perturbation:g}:",
          finite_difference_gradient(decision, step=perturbation))
partial_figure = show_partial_derivatives()
_pyplot().show()
_pyplot().close(partial_figure)

### 3 · Directional derivatives and level-set geometry

For a differentiable score and a small displacement \\(\Delta u\\), the first-order prediction is

> $\displaystyle J(u+\Delta u;\lambda_E)-J(u;\lambda_E)\approx\nabla_u J(u;\lambda_E)^{\mathsf T}\Delta u.$

For a unit direction \\(d\\) and a small positive step length \\(\alpha\\), set \\(\Delta u=\alpha d\\). The directional derivative, the score's rate of change along \\(d\\), is \\(\nabla_u J^{\mathsf T}d\\).

| Inner product with the gradient | Local conclusion |
|:---|:---|
| Positive | A sufficiently small positive step increases the score |
| Negative | A sufficiently small positive step decreases the score |
| Zero | The first-order term vanishes; finite steps can still change the score |

If \\(u(s)\\) traces a differentiable equal-score curve, the chain rule gives

> $\displaystyle \frac{d}{ds}J(u(s);\lambda_E)=\nabla_u J^{\mathsf T}\frac{du}{ds}=0.$

Thus a nonzero gradient is perpendicular to the tangent of a regular level curve. A finite move along a straight tangent line generally leaves the curved level set.

Under the Euclidean size rule \\(\lVert d\rVert_2=1\\), the Cauchy–Schwarz inequality gives

> $\displaystyle -\lVert\nabla_u J\rVert_2\le\nabla_u J^{\mathsf T}d\le\lVert\nabla_u J\rVert_2.$

When the gradient is nonzero, the steepest local decrease therefore uses

> $\displaystyle d_{\mathrm{down}}=-\frac{\nabla_u J}{\lVert\nabla_u J\rVert_2}.$

This assumes equal Euclidean step lengths and all local directions available. Another norm or an active constraint can change the permitted steepest direction. If the gradient is zero, normalization is undefined; a stationary point alone does not establish a minimum for a general problem.

The arrows are normalized to show direction, so their lengths do not represent gradient magnitude. Descent points toward lower score levels; the tangent is perpendicular to the gradient at the current decision.

The score contours use the same finite-grid evaluation as Part 4. Normalized gradient and tangent vectors describe the local geometry at a fixed decision.

In [ ]:
def evaluate_grid(points_per_axis=81, energy_weight=ENERGY_WEIGHT):
    levels = np.linspace(MIN_COOLING, MAX_COOLING, points_per_axis)
    early, late = np.meshgrid(levels, levels)
    records = [evaluate_candidate((e, l), energy_weight)
               for e, l in zip(early.ravel(), late.ravel())]
    return {
        "early": early, "late": late, "records": records,
        "scores": np.array([r["objective"] for r in records]).reshape(early.shape),
        "energy": np.array([r["energy"] for r in records]).reshape(early.shape),
        "feasible": np.array([r["feasible"] for r in records]).reshape(early.shape),
        "energy_weight": energy_weight,
    }


def draw_score_map(axis, grid):
    axis.contourf(grid["early"], grid["late"], grid["feasible"].astype(int),
                  levels=[-0.5, 0.5, 1.5], colors=[GRAY, TEAL], alpha=0.16)
    contours = axis.contour(grid["early"], grid["late"], grid["scores"],
                            levels=[53, 55, 60, 75, 100, 150, 250, 400],
                            colors=BLUE, linewidths=1.0, alpha=0.85)
    # Place labels away from plot boundaries and the upper-right legend.
    label_positions = [(55, (3.25, 1.25)), (60, (3.9, 1.4)),
                       (75, (4.5, 2.2)), (100, (2.0, 1.1)),
                       (150, (1.7, 0.65)), (250, (1.0, 0.5)),
                       (400, (0.2, 0.3))]
    for level, position in label_positions:
        axis.clabel(contours, levels=[level], manual=[position], fontsize=8, fmt="%g")
    decision_axes(axis)


classroom_grid = evaluate_grid()

In [ ]:
def show_gradient_geometry(grid, decision=(3.0, 2.0)):
    plt = _pyplot()
    decision = np.asarray(decision, dtype=float)
    gradient = finite_difference_gradient(decision)
    magnitude = np.linalg.norm(gradient)
    figure, axis = plt.subplots(figsize=(7.8, 6.0))
    draw_score_map(axis, grid)
    axis.contour(grid["early"], grid["late"], grid["scores"],
                  levels=[score(decision)], colors=["#172033"], linewidths=2)
    if magnitude > 1e-10:
        uphill = gradient / magnitude
        tangent = np.array([-uphill[1], uphill[0]])
        for direction, color in [(uphill, BLUE), (-uphill, ORANGE)]:
            axis.annotate("", xy=decision + 0.8 * direction, xytext=decision,
                          arrowprops=dict(arrowstyle="->", lw=2.8, color=color))
        ends = decision + np.array([-0.8, 0.8])[:, None] * tangent
        axis.plot(*ends.T, "--", color="#172033", lw=1.5)
    axis.scatter(*decision, s=90, color=ORANGE, edgecolor="white", zorder=5)
    axis.legend(handles=[Line2D([], [], color=BLUE, lw=2, label="Normalized gradient"),
                         Line2D([], [], color=ORANGE, lw=2, label="Normalized descent"),
                         Line2D([], [], color="#172033", linestyle="--", label="Local tangent"),
                         Patch(color=TEAL, alpha=0.2, label="Feasible region")],
                 loc="upper right", fontsize=9)
    axis.set_title("The gradient is normal to the local equal-score curve")
    figure.tight_layout()
    return figure

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-2_mathematics_for_optimization/assets/04_level_sets_and_gradient.svg" alt="Score contours with a normalized gradient, descent direction, and local tangent" width="760" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

In [ ]:
gradient_figure = show_gradient_geometry(classroom_grid)
_pyplot().show()
_pyplot().close(gradient_figure)

decision = np.array([3.0, 2.0])
gradient = finite_difference_gradient(decision)
uphill = gradient / np.linalg.norm(gradient)
tangent = np.array([-uphill[1], uphill[0]])
for name, direction in [("uphill", uphill), ("tangent", tangent), ("downhill", -uphill)]:
    print(f"{name:8s}: unit length={np.linalg.norm(direction):.3f}, "
          f"directional derivative={gradient @ direction:+.6f}")

### 4 · Gradient descent and step length

The ordinary gradient-descent update uses a learning rate \\(\gamma>0\\):

> $\displaystyle u^{(k+1)}=u^{(k)}-\gamma\nabla_u J(u^{(k)};\lambda_E).$

The superscript \\(k\\) counts search iterations, not classroom time steps \\(t\\). The Euclidean length of this update is \\(\gamma\lVert\nabla_u J\rVert_2\\).

The normalized direction from Section 3 gives an update whose length is explicit:

> $\displaystyle u_{\mathrm{trial}}=u+\alpha d_{\mathrm{down}},\qquad \lVert u_{\mathrm{trial}}-u\rVert_2=\alpha.$

Here \\(\alpha\\) is a length in cooling units. When the gradient is nonzero, this matches an ordinary update with \\(\gamma=\alpha/\lVert\nabla_u J\rVert_2\\). The calculation evaluates a single trial step; convergence of an iterative method is outside its scope.

For \\(u=(3,2)\\) and \\(\lambda_E=1\\), step lengths 0.1, 0.5, and 2.0 give three trials along the same normalized descent direction. Feasibility and score improvement are evaluated separately.

The evaluator rejects infeasible trials first. It compares scores only when both the start and trial are feasible. Numerical score changes at rejected points are shown only to explain the geometry.

In [ ]:
def evaluate_gradient_trial(decision, length, energy_weight=ENERGY_WEIGHT):
    if length < 0:
        raise ValueError("Step length must be nonnegative.")
    start = evaluate_candidate(decision, energy_weight)
    gradient = finite_difference_gradient(decision, energy_weight)
    magnitude = np.linalg.norm(gradient)
    direction = -gradient / magnitude if magnitude > 1e-10 else np.zeros(2)
    trial = evaluate_candidate(start["decision"] + length * direction, energy_weight)
    if not start["feasible"]:
        status = "Infeasible start: descent comparison undefined."
    elif not trial["feasible"]:
        status = "Reject: " + ", ".join(trial["violations"])
    elif magnitude <= 1e-10:
        status = "Stationary: no normalized gradient direction."
    elif length == 0:
        status = "No move."
    elif trial["objective"] < start["objective"]:
        status = "Acceptable improvement for this trial."
    else:
        status = "Feasible trial: no score improvement."
    return {"start": start, "trial": trial, "gradient": gradient, "direction": direction,
            "predicted_change": float(length * (gradient @ direction)),
            "actual_change": trial["objective"] - start["objective"], "status": status}


def show_gradient_explorer(grid):
    plt = _pyplot(interactive=True)
    from matplotlib.widgets import Slider
    figure, axes = plt.subplots(1, 2, figsize=(11.6, 7.3))
    figure.subplots_adjust(left=0.075, right=0.97, top=0.90, bottom=0.39, wspace=0.32)
    sliders = [
        Slider(figure.add_axes([0.20, 0.26, 0.62, 0.027]), "Early cooling", 0, 5,
               valinit=3, valstep=0.05, color=BLUE),
        Slider(figure.add_axes([0.20, 0.205, 0.62, 0.027]), "Late cooling", 0, 5,
               valinit=2, valstep=0.05, color=BLUE),
        Slider(figure.add_axes([0.20, 0.15, 0.62, 0.027]), "Step length", 0, 2.5,
               valinit=0.1, valstep=0.05, color=PURPLE),
    ]
    note = figure.text(0.075, 0.022, "", fontsize=9, linespacing=1.5)
    lengths = np.linspace(0, 2.5, 101)
    cached = {}
    def update(_):
        decision = np.array([sliders[0].val, sliders[1].val])
        length = sliders[2].val
        comparison = evaluate_gradient_trial(decision, length)
        start, trial = comparison["start"], comparison["trial"]
        direction, gradient = comparison["direction"], comparison["gradient"]
        key = tuple(decision)
        if cached.get("decision") != key:
            results = [evaluate_candidate(decision + amount * direction) for amount in lengths]
            cached.update(decision=key,
                          scores=np.array([r["objective"] for r in results]),
                          feasible=np.array([r["feasible"] for r in results]))
        for axis in axes:
            axis.clear()
        draw_score_map(axes[0], grid)
        axes[0].plot([decision[0], trial["decision"][0]],
                      [decision[1], trial["decision"][1]], color=ORANGE, lw=2)
        axes[0].scatter(*decision, color=BLUE if start["feasible"] else GRAY,
                         marker="o" if start["feasible"] else "X", s=65, label="Start", zorder=4)
        axes[0].scatter(*trial["decision"], color=ORANGE if trial["feasible"] else GRAY,
                         marker="o" if trial["feasible"] else "X", s=90, label="Trial", zorder=5)
        # Keep an out-of-bounds trial visible so rejection can be diagnosed.
        axes[0].set(xlim=(min(-0.1, trial["decision"][0] - 0.2), max(5.1, trial["decision"][0] + 0.2)),
                    ylim=(min(-0.1, trial["decision"][1] - 0.2), max(5.1, trial["decision"][1] + 0.2)),
                    title="Direction and feasibility")
        axes[0].legend(loc="upper right", fontsize=8)
        actual, feasible = cached["scores"], cached["feasible"]
        predicted = start["objective"] + lengths * (gradient @ direction)
        axes[1].plot(lengths, np.where(~feasible, actual, np.nan), color=GRAY, lw=2,
                      label="Infeasible score extension")
        axes[1].plot(lengths, np.where(feasible, actual, np.nan), color=BLUE, lw=2.5,
                      label="Feasible actual scores")
        axes[1].plot(lengths, predicted, "--", color=ORANGE, label="First-order prediction")
        axes[1].scatter(length, trial["objective"], color=ORANGE if trial["feasible"] else GRAY,
                         marker="o" if trial["feasible"] else "X", s=70, zorder=5)
        axes[1].axvline(length, color=PURPLE, alpha=0.55)
        axes[1].set(xlabel="Step length (cooling units)", ylabel="Score J (score units)",
                    title="A local prediction can fail for a large step", xlim=(-0.03, 2.53))
        axes[1].legend(fontsize=8)
        style_axis(axes[1])
        note.set_text(f"{comparison['status']}\n"
                      f"Trial u=({trial['decision'][0]:.3f}, {trial['decision'][1]:.3f}); "
                      f"D={trial['discomfort']:.2f}, E={trial['energy']:.2f}; "
                      f"predicted change={comparison['predicted_change']:+.3f}, "
                      f"actual change={comparison['actual_change']:+.3f}.\n"
                      "Fixed energy weight = 1. Gray scores explain the geometry; rejected trials are not selected.")
        figure.canvas.draw_idle()
        figure._comparison = comparison
    for slider in sliders:
        slider.on_changed(update)
    figure._sliders, figure._update = sliders, update
    update(None)
    return figure

Each row uses the same starting decision and normalized descent direction. Only the step length changes.


In [ ]:
for length in [0.1, 0.5, 2.0]:
    comparison = evaluate_gradient_trial((3.0, 2.0), length)
    print(f"length={length:.1f}: {comparison['status']}")
    print(f"  predicted change={comparison['predicted_change']:+.3f}, "
          f"actual change={comparison['actual_change']:+.3f}")

For a fixed starting decision, the left panel locates the trial along the descent direction. The right panel compares the first-order score prediction with simulation over the same line. The controls parameterize the starting decision and step length.


<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-2_mathematics_for_optimization/assets/07_gradient_step_explorer.svg" alt="Cooling-decision and score-versus-step-length panels with sliders and feasibility status" width="900" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

In [ ]:
gradient_explorer = show_gradient_explorer(classroom_grid)
_pyplot().show()

At \\(u=(3,2)\\), \\(\nabla_u J\approx(-5.152,0.842)\\). A short descent step increases early cooling and slightly reduces late cooling.

| Step length | Predicted change | Simulated change | Decision |
|:---|---:|---:|:---|
| 0.1 | About $-0.522$ | About $-0.298$ | Feasible improvement |
| 0.5 | About $-2.610$ | About $+1.930$ | Feasible, but worse |
| 2.0 | About $-10.441$ | About $+36.614$ | Infeasible; reject |

A negative directional derivative predicts a decrease for sufficiently small positive steps. It does not guarantee an improvement for every step length. Feasibility and objective decrease are separate tests.

For a general differentiable unconstrained problem, an interior local minimum must have zero gradient. Zero gradient alone is insufficient, and a constrained minimum can have a nonzero gradient. At an active boundary, the steepest unconstrained direction may leave the feasible set. These facts motivate later search methods and constrained optimization.

### 5 · Scope of the local analysis

The derivative is taken with respect to the chosen cooling coordinates. Final temperature remains a system state produced by the simulation; it is not an independent decision coordinate.

| Mathematical object | What it establishes | Limit |
|:---|:---|:---|
| Displacement norm | Size under the specified rule | A short step can violate a physical requirement |
| Zero gradient–direction inner product | Zero first-order score change | A finite straight step can leave the level set |
| Equal-score contour | Equal scalar score | Discomfort, energy, and state paths can differ |
| Negative directional derivative | Local decrease for sufficiently small positive steps | A larger step can increase the score or be infeasible |

At fixed \\(u\\), changing \\(\lambda_E\\) preserves the temperature path, \\(D(u)\\), \\(E(u)\\), and feasibility. It changes the score and generally the gradient:

> $\displaystyle \nabla_u J(u;\lambda_E)=\nabla_u D(u)+\lambda_E\nabla_u E(u).$

Consequently, the descent direction and selected decision can change even though the physical outcome of a fixed decision does not.

If the formulation also includes a binary ventilation switch, ordinary derivatives apply to the continuous cooling coordinates with the switch held fixed. The allowed switch values require discrete comparison; a fractional switch is not an admissible decision.
